# BioSentinelle Normandie — V3 | Détection, revue humaine, classification et vidéos annotées

**Notebook Google Colab autonome** — 11 espèces, extraction par vidéo, MegaDetector officiel, revue manuelle des recadrages, entraînement en deux phases, évaluation par vidéo et inférence MP4.

**Ordre :** exécuter les cellules dans l’ordre. Les données et checkpoints sont enregistrés dans Google Drive. **Activez un GPU** (Exécution > Modifier le type d’exécution).

> Objectif de 80–90 % = cible expérimentale, non promesse. Les vidéos d’une même séquence ne doivent pas se retrouver dans plusieurs ensembles. Une vidéo dont le nom indique l’espèce n’est pas une annotation de chaque animal visible.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%pip -q install "megadetector" "timm>=1.0" "scikit-learn" "ipywidgets" "opencv-python-headless" "pillow" "tqdm" "matplotlib" "pandas" "seaborn"

In [ ]:
import os, sys, json, random, re, hashlib, shutil, warnings, gc
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np, pandas as pd, cv2, torch, timm
from PIL import Image, ImageDraw
from tqdm.auto import tqdm
from IPython.display import display, clear_output, Video
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from google.colab import drive, files
import ipywidgets as widgets

drive.mount('/content/drive')
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
ROOT=Path('/content/drive/MyDrive/BioSentinelle')
VIDEOS_DIR=ROOT/'videos'
OUT=ROOT/'resultats_v3'; OUT.mkdir(parents=True,exist_ok=True)
WORK=Path('/content/biosentinelle_v3'); WORK.mkdir(parents=True,exist_ok=True)
FRAMES=WORK/'frames'; FRAMES.mkdir(exist_ok=True)
CROPS=WORK/'crops'; CROPS.mkdir(exist_ok=True)
CLASSES=['chevreuil','sanglier','buse','pigeons','martres','renards','blaireaux','chouette','geai','faisans']
LATIN={'chevreuil':'Capreolus capreolus','sanglier':'Sus scrofa','buse':'Buteo buteo','pigeons':'Columba palumbus','martres':'Martes martes','renards':'Vulpes vulpes','blaireaux':'Meles meles','chouette':'Strix aluco','geai':'Garrulus glandarius','faisans':'Phasianus colchicus'}
CLASS_TO_IDX={s:i for i,s in enumerate(CLASSES)}
print('GPU:',DEVICE,'| Vidéos:',VIDEOS_DIR,'| Résultats:',OUT)
assert VIDEOS_DIR.exists(),f'Dossier manquant : {VIDEOS_DIR}'

## 1 — Inventaire et séparation stricte par vidéo
Les noms de fichiers doivent contenir l’espèce (ex. `geai_004.mp4`). Pour les classes très rares, un ensemble de validation/test par vidéo reste statistiquement fragile.

In [ ]:
VIDEO_EXT={'.mp4','.mov','.avi','.mkv','.m4v'}
def species_from_path(p):
    text=' '.join([p.stem]+[q.name for q in list(p.parents)[:2]]).lower()
    hits=[s for s in CLASSES if re.search(r'(?<![a-z])'+re.escape(s)+r'(?![a-z])',text)]
    return hits[0] if len(hits)==1 else None
videos=[]; ignored=[]
for p in sorted(VIDEOS_DIR.rglob('*')):
    if p.is_file() and p.suffix.lower() in VIDEO_EXT:
        s=species_from_path(p)
        (videos if s else ignored).append({'video_path':str(p),'species':s} if s else str(p))
assert videos,'Aucune vidéo avec espèce reconnue dans le nom.'
df_videos=pd.DataFrame(videos)
print('Vidéos:',len(df_videos),'| Ignorées:',len(ignored))
print(df_videos.species.value_counts().reindex(CLASSES,fill_value=0).to_string())
if ignored: print('Exemples ignorés:',ignored[:8])
# Séparation par vidéo, chaque classe est traitée séparément.
# 1 vidéo : train uniquement ; 2 vidéos : train/test ; >=3 : train/val/test si possible.
def split_species(g):
    ids=g.index.to_list(); random.Random(SEED+CLASSES.index(g.species.iloc[0])).shuffle(ids)
    count=len(ids)
    if count==1: nv,nt=0,0
    elif count==2: nv,nt=0,1
    else: nv=max(1,round(count*.15)); nt=max(1,round(count*.15))
    if nv+nt>=count: nv=max(0,count-nt-1)
    for idx in ids[:nt]: df_videos.loc[idx,'split']='test'
    for idx in ids[nt:nt+nv]: df_videos.loc[idx,'split']='val'
    for idx in ids[nt+nv:]: df_videos.loc[idx,'split']='train'
for _,group in df_videos.groupby('species',sort=False): split_species(group)
df_videos.to_csv(OUT/'videos_split.csv',index=False)
print('\nRépartition :\n',pd.crosstab(df_videos.species,df_videos.split).reindex(CLASSES,fill_value=0))

## 2 — Extraction des images (cache local + index persistant)
Ne pas extraire chaque frame : choisir des images espacées, en évitant de dépasser le quota par vidéo. L’index et les images peuvent être régénérés si la session Colab redémarre.

In [ ]:
FRAMES_PER_VIDEO=24
MIN_SECONDS_BETWEEN_FRAMES=0.8
frame_records=[]
for vi,row in tqdm(df_videos.iterrows(),total=len(df_videos),desc='Vidéos'):
    cap=cv2.VideoCapture(row.video_path)
    if not cap.isOpened(): print('Illisible:',row.video_path); continue
    fps=cap.get(cv2.CAP_PROP_FPS); fps=fps if fps and np.isfinite(fps) and fps>0 else 25
    count=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); duration=count/fps if count>0 else 0
    n=max(1,min(FRAMES_PER_VIDEO,int(duration/MIN_SECONDS_BETWEEN_FRAMES)+1))
    frame_ids=np.unique(np.linspace(0,max(0,count-1),n,dtype=int))
    vid_id=hashlib.sha1(row.video_path.encode()).hexdigest()[:14]
    folder=FRAMES/vid_id;folder.mkdir(exist_ok=True)
    for frame_i in frame_ids:
        dest=folder/f'{int(frame_i):07d}.jpg'
        if not dest.exists():
            cap.set(cv2.CAP_PROP_POS_FRAMES,int(frame_i)); ok,img=cap.read()
            if not ok:continue
            cv2.imwrite(str(dest),img,[cv2.IMWRITE_JPEG_QUALITY,90])
        frame_records.append({'video_path':row.video_path,'video_id':vid_id,'species':row.species,'split':row.split,'frame_idx':int(frame_i),'time_sec':float(frame_i/fps),'frame_path':str(dest)})
    cap.release()
frames_df=pd.DataFrame(frame_records)
assert len(frames_df),'Aucune image extraite.'
frames_df.to_csv(WORK/'frames_index.csv',index=False)
print('Images:',len(frames_df),'| Vidéos:',frames_df.video_id.nunique())

## 3 — MegaDetector officiel (sans SpeciesNet/Kaggle)
Utilise **`megadetector.detection.run_detector.load_detector('MDV5A')`**, API officielle. Les poids sont téléchargés automatiquement au premier appel. **Pas de substitution silencieuse** par un détecteur COCO. Commencez par le test visuel de quelques images avant de traiter le dataset complet.

In [ ]:
from megadetector.detection import run_detector
from megadetector.visualization import visualization_utils as vis_utils
DETECTOR_NAME='MDV5A'
DETECT_CONF=0.10  # garder les candidats, filtrer plus tard après inspection
print('Chargement',DETECTOR_NAME,'— premier téléchargement potentiellement long')
try:
    detector=run_detector.load_detector(DETECTOR_NAME)
except Exception as e:
    raise RuntimeError('Chargement MegaDetector impossible. Vérifier installation, connexion et versions torch/YOLOv5. Aucun détecteur de remplacement n’est appliqué.') from e

def detect_image(path):
    image=vis_utils.load_image(str(path))
    result=detector.generate_detections_one_image(image)
    return [d for d in result.get('detections',[]) if str(d.get('category'))=='1' and float(d.get('conf',0))>=DETECT_CONF]

def draw_detections(path,detections):
    im=Image.open(path).convert('RGB'); w,h=im.size; d=ImageDraw.Draw(im)
    for x in detections:
        a,b,c,e=x['bbox']; box=[int(a*w),int(b*h),int((a+c)*w),int((b+e)*h)]
        d.rectangle(box,outline='lime',width=3); d.text((box[0],max(0,box[1]-16)),f"Animal {x['conf']:.2f}",fill='lime')
    return im
sample=frames_df.sample(min(6,len(frames_df)),random_state=SEED)
fig,axes=plt.subplots(len(sample),1,figsize=(10,4*len(sample)),squeeze=False)
for ax,(_,r) in zip(axes.ravel(),sample.iterrows()):
    dets=detect_image(r.frame_path);ax.imshow(draw_detections(r.frame_path,dets));ax.set_title(f'{r.species} — {len(dets)} animal(aux)');ax.axis('off')
plt.tight_layout();plt.show()

In [ ]:
# Après validation visuelle, lancer la détection de toutes les images.
DETECTION_CACHE=OUT/'detections_v3.json'
# Ne pas réutiliser un cache issu d'un autre index de frames.
all_paths=set(frames_df.frame_path)
if DETECTION_CACHE.exists():
    cache=json.loads(DETECTION_CACHE.read_text())
    det_map={k:v for k,v in cache.items() if k in all_paths}
else: det_map={}
for i,path in enumerate(tqdm(frames_df.frame_path.unique(),desc='MegaDetector')):
    if path not in det_map:
        try: det_map[path]=detect_image(path)
        except Exception as e: print('Détection impossible:',path,str(e)[:160]); det_map[path]=[]
    if (i+1)%100==0: DETECTION_CACHE.write_text(json.dumps(det_map))
DETECTION_CACHE.write_text(json.dumps(det_map))
print('Images analysées:',len(det_map),'| avec animal:',sum(bool(v) for v in det_map.values()))
print('Sans détection:',sum(not v for v in det_map.values()),'— examiner avant entraînement.')

MegaDetector:   0%|          | 0/5386 [00:00<?, ?it/s]

## 4 — Recadrages propres + contrôle qualité
Pas de recadrage « image entière » en remplacement d’une détection ratée. Une vidéo peut contenir plusieurs animaux : **les labels dérivés du nom de vidéo sont provisoires**. Les mauvaises détections sont écartées lors de la revue.

In [ ]:
MIN_CROP_SIDE=28
CROP_MARGIN=0.12
MAX_CROPS_PER_FRAME=4

def crop_xywh(frame_path,bbox,output_path,margin=CROP_MARGIN):
    im=Image.open(frame_path).convert('RGB'); W,H=im.size
    x,y,w,h=[float(z) for z in bbox]
    x1=max(0,int((x-w*margin)*W));y1=max(0,int((y-h*margin)*H))
    x2=min(W,int((x+w*(1+margin))*W));y2=min(H,int((y+h*(1+margin))*H))
    if x2-x1<MIN_CROP_SIDE or y2-y1<MIN_CROP_SIDE:return False
    output_path=Path(output_path);output_path.parent.mkdir(parents=True,exist_ok=True)
    im.crop((x1,y1,x2,y2)).save(output_path,quality=93)
    return True

crop_rows=[]
for _,r in tqdm(frames_df.iterrows(),total=len(frames_df),desc='Recadrages'):
    ds=sorted(det_map.get(r.frame_path,[]),key=lambda d:d.get('conf',0),reverse=True)[:MAX_CROPS_PER_FRAME]
    for j,d in enumerate(ds):
        key=hashlib.sha1((r.frame_path+str(j)).encode()).hexdigest()[:20]
        path=CROPS/f'{key}.jpg'
        if crop_xywh(r.frame_path,d['bbox'],path):
            crop_rows.append({'crop_id':key,'crop_path':str(path),'frame_path':r.frame_path,'video_id':r.video_id,'video_path':r.video_path,'split':r.split,'species':r.species,'conf':float(d['conf']),'bbox':json.dumps(d['bbox']),'review':'pending','review_label':''})
crops_df=pd.DataFrame(crop_rows)
assert len(crops_df),'MegaDetector n’a produit aucun recadrage exploitable. Vérifier la galerie de détection et les seuils.'
print('Recadrages:',len(crops_df),'| Répartition :\n',crops_df.species.value_counts().to_string())
fig,axes=plt.subplots(3,5,figsize=(14,9));axes=axes.ravel()
for ax,(_,r) in zip(axes,crops_df.sample(min(15,len(crops_df)),random_state=SEED).iterrows()):
    ax.imshow(Image.open(r.crop_path));ax.set_title(f"{r.species} ({r.conf:.2f})",fontsize=9);ax.axis('off')
for ax in axes[len(crops_df.head(15)):]:ax.axis('off')
plt.tight_layout();plt.show()

## 5 — Revue humaine interactive : confirmer, corriger, rejeter **et recadrer**
La revue est enregistrée sur Drive (`review_v3.csv`). Utilisez les curseurs pour modifier la boîte normalisée **x, y, largeur, hauteur**. L’image recadrée est régénérée lorsque vous cliquez sur **Valider / Corriger**. Les corrections ne modifient jamais les ensembles train/val/test.

In [ ]:
REVIEW_FILE=OUT/'review_v3.csv'
if REVIEW_FILE.exists():
    old=pd.read_csv(REVIEW_FILE).fillna('')
    old=old.drop_duplicates('crop_id',keep='last').set_index('crop_id')
    for col in ['review','review_label','bbox']:
        if col in old.columns: crops_df[col]=crops_df.crop_id.map(old[col]).fillna(crops_df[col])
else: old=None

def persist_review():
    crops_df[['crop_id','review','review_label','bbox']].to_csv(REVIEW_FILE,index=False)

# Privilégier les classes peu représentées, puis les détections incertaines.
counts=crops_df.species.value_counts()
order=sorted(crops_df.index,key=lambda i:(counts[crops_df.loc[i,'species']],crops_df.loc[i,'conf']))
remaining=[i for i in order if crops_df.loc[i,'review']=='pending']
print('En attente:',len(remaining),'| Déjà revus:',len(crops_df)-len(remaining))

viewer=widgets.Image(format='jpeg',width=560)
info=widgets.HTML()
choice=widgets.Dropdown(options=CLASSES,description='Espèce:')
sliders={name:widgets.FloatSlider(value=0,min=0,max=1,step=.005,description=name,readout_format='.3f',continuous_update=False,layout=widgets.Layout(width='400px')) for name in ['x','y','w','h']}
buttons=[widgets.Button(description=t,button_style=style) for t,style in [('Valider / Corriger','success'),('Rejeter','danger'),('Passer','')]]
position={'i':0}

def refresh(*_):
    if position['i']>=len(remaining):info.value='<b>Revue terminée pour cette session.</b>';viewer.value=b'';return
    idx=remaining[position['i']];r=crops_df.loc[idx];choice.value=r.species
    bbox=json.loads(r.bbox)
    for k,v in zip(['x','y','w','h'],bbox):sliders[k].value=float(v)
    viewer.value=Path(r.crop_path).read_bytes()
    info.value=f"<b>{position['i']+1}/{len(remaining)}</b> — {r.species} — conf {r.conf:.2f} — vidéo {Path(r.video_path).name}"

def action(kind):
    if position['i']>=len(remaining):return
    idx=remaining[position['i']];r=crops_df.loc[idx]
    if kind=='reject':crops_df.loc[idx,'review']='reject';crops_df.loc[idx,'review_label']=''
    elif kind=='accept':
        bb=[sliders[k].value for k in ['x','y','w','h']]
        if bb[2]<=0 or bb[3]<=0 or bb[0]+bb[2]>1 or bb[1]+bb[3]>1:
            info.value='<b>Boîte invalide : vérifier x+w et y+h (≤1).</b>';return
        if not crop_xywh(r.frame_path,bb,r.crop_path):info.value='<b>Recadrage trop petit.</b>';return
        crops_df.loc[idx,'bbox']=json.dumps(bb)
        crops_df.loc[idx,'review']='accept';crops_df.loc[idx,'review_label']=choice.value
    persist_review();position['i']+=1;refresh()
buttons[0].on_click(lambda _:action('accept'))
buttons[1].on_click(lambda _:action('reject'))
buttons[2].on_click(lambda _:action('skip'))
display(widgets.VBox([info,viewer,choice,*sliders.values(),widgets.HBox(buttons)]))
refresh()
print('Astuce : commencez par les espèces rares et les recadrages douteux. Les non revus restent des labels faibles.')

## 6 — Manifeste final et qualité du dataset
Les recadrages rejetés sont exclus. Les labels corrigés sont appliqués. **Ne pas évaluer uniquement sur des labels vidéo non vérifiés** : la mesure serait bruitée. Limiter les recadrages par vidéo réduit les doublons.

In [ ]:
if REVIEW_FILE.exists():
    old=pd.read_csv(REVIEW_FILE).fillna('').drop_duplicates('crop_id',keep='last').set_index('crop_id')
    for col in ['review','review_label','bbox']:
        if col in old: crops_df[col]=crops_df.crop_id.map(old[col]).fillna(crops_df[col])
manifest=crops_df[crops_df.review!='reject'].copy()
manifest['label']=np.where((manifest.review=='accept')&(manifest.review_label.isin(CLASSES)),manifest.review_label,manifest.species)
MAX_CROPS_PER_VIDEO=18
parts=[]
for (_, _),g in manifest.groupby(['video_id','split']):
    parts.append(g.sort_values(['review','conf'],ascending=[True,False]).head(MAX_CROPS_PER_VIDEO))
manifest=pd.concat(parts,ignore_index=True)
manifest.to_csv(OUT/'manifest_v3.csv',index=False)
print('Total:',len(manifest),'| Validés humainement:',int((manifest.review=='accept').sum()))
print(pd.crosstab(manifest.label,manifest.split).reindex(CLASSES,fill_value=0))
print('Attention : si une classe est absente de train, il faut ajouter des vidéos avant de poursuivre.')
missing=set(CLASSES)-set(manifest[manifest.split=='train'].label)
if missing:raise RuntimeError('Classes sans exemple train: '+', '.join(sorted(missing)))

## 7 — Classification en deux phases
Backbone ImageNet `convnext_tiny` (modifiable), augmentations réalistes, poids de classe modérés, sélection sur **macro-F1 validation**, sauvegarde du meilleur modèle. `num_workers=0` pour éviter les erreurs multiprocessing Colab.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
MODEL_NAME='convnext_tiny'
IMG_SIZE=256
BATCH_SIZE=24
EPOCHS_HEAD=6
EPOCHS_FINETUNE=18
PATIENCE=6
CHECKPOINT=OUT/'classifier_v3_best.pt'
train_tf=transforms.Compose([transforms.RandomResizedCrop(IMG_SIZE,scale=(.72,1.0)),transforms.RandomHorizontalFlip(),transforms.ColorJitter(.2,.2,.2,.05),transforms.RandomApply([transforms.GaussianBlur(3)],p=.12),transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
val_tf=transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)),transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
class CropsDS(Dataset):
    def __init__(self,df,tf):self.df=df.reset_index(drop=True);self.tf=tf
    def __len__(self):return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]
        with Image.open(r.crop_path) as img: x=self.tf(img.convert('RGB'))
        return x,CLASS_TO_IDX[r.label]
train_df=manifest[manifest.split=='train'].copy();val_df=manifest[manifest.split=='val'].copy();test_df=manifest[manifest.split=='test'].copy()
assert len(train_df) and len(val_df),'Train ou validation vide : ajoutez des vidéos.'
train_ds=CropsDS(train_df,train_tf);val_ds=CropsDS(val_df,val_tf);test_ds=CropsDS(test_df,val_tf)
counts=train_df.label.value_counts()
weights=np.array([1/np.sqrt(counts[s]) for s in train_df.label],dtype=float)
weights=np.where(train_df.review.eq('accept'),weights,weights*.8)
sampler=WeightedRandomSampler(torch.as_tensor(weights,dtype=torch.double),num_samples=len(train_df),replacement=True)
train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,sampler=sampler,num_workers=0,pin_memory=DEVICE=='cuda')
val_loader=DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=DEVICE=='cuda')
test_loader=DataLoader(test_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=0)
model=timm.create_model(MODEL_NAME,pretrained=True,num_classes=len(CLASSES)).to(DEVICE)
loss_fn=nn.CrossEntropyLoss(label_smoothing=.07)
print('Train',len(train_ds),'Val',len(val_ds),'Test',len(test_ds))

In [ ]:
def freeze_backbone(m):
    for p in m.parameters():p.requires_grad=False
    for p in m.get_classifier().parameters():p.requires_grad=True

def unfreeze_all(m):
    for p in m.parameters():p.requires_grad=True

@torch.no_grad()
def evaluate(m,loader):
    m.eval();ys=[];ps=[];losses=[]
    for x,y in loader:
        x=x.to(DEVICE);y=y.to(DEVICE);logits=m(x)
        losses.append(float(loss_fn(logits,y)));ys.extend(y.cpu().tolist());ps.extend(logits.argmax(1).cpu().tolist())
    return {'loss':float(np.mean(losses)) if losses else np.nan,'acc':accuracy_score(ys,ps) if ys else 0,'macro_f1':f1_score(ys,ps,labels=list(range(len(CLASSES))),average='macro',zero_division=0) if ys else 0}

best_score=-1;history=[]
def train_phase(phase,epochs,lr,patience):
    global best_score
    opt=torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),lr=lr,weight_decay=.03)
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=max(1,epochs))
    stale=0
    for epoch in range(1,epochs+1):
        model.train();running=[]
        for x,y in tqdm(train_loader,desc=f'{phase} {epoch}/{epochs}',leave=False):
            x=x.to(DEVICE);y=y.to(DEVICE);opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda',enabled=DEVICE=='cuda'):
                logits=model(x);loss=loss_fn(logits,y)
            loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.0);opt.step();running.append(float(loss.detach()))
        scheduler.step();metrics=evaluate(model,val_loader)
        row={'phase':phase,'epoch':epoch,'train_loss':np.mean(running),**metrics};history.append(row)
        print(f"{phase} {epoch}: train_loss={row['train_loss']:.3f} val_acc={metrics['acc']:.3f} val_macroF1={metrics['macro_f1']:.3f}")
        if metrics['macro_f1']>best_score+1e-5:
            best_score=metrics['macro_f1'];stale=0
            torch.save({'state_dict':model.state_dict(),'classes':CLASSES,'model_name':MODEL_NAME,'img_size':IMG_SIZE,'val_macro_f1':best_score},CHECKPOINT)
            print('  ✓ meilleur checkpoint sauvegardé')
        else:
            stale+=1
            if stale>=patience:print('Early stopping');break
    pd.DataFrame(history).to_csv(OUT/'history_v3.csv',index=False)

freeze_backbone(model)
train_phase('head',EPOCHS_HEAD,8e-4,patience=EPOCHS_HEAD+1)
unfreeze_all(model)
train_phase('finetune',EPOCHS_FINETUNE,2e-5,PATIENCE)
print('Meilleur macro-F1 validation:',best_score)

## 8 — Évaluation indépendante : recadrage ET vidéo
La prédiction vidéo est la moyenne des probabilités sur ses recadrages. Les classes absentes du test sont signalées ; leurs scores ne peuvent pas être interprétés comme une mesure fiable.

In [ ]:
checkpoint=torch.load(CHECKPOINT,map_location=DEVICE,weights_only=False)
model.load_state_dict(checkpoint['state_dict']);model.eval()
@torch.no_grad()
def probabilities(ds):
    loader=DataLoader(ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=0)
    out=[]
    for x,_ in tqdm(loader,desc='Évaluation',leave=False):
        p=model(x.to(DEVICE)).softmax(-1).cpu().numpy();out.extend(p)
    return np.asarray(out)
if len(test_df):
    probs=probabilities(test_ds)
    pred=probs.argmax(axis=1);truth=test_df.label.map(CLASS_TO_IDX).to_numpy()
    print('=== Par recadrage ===')
    print(classification_report(truth,pred,labels=list(range(len(CLASSES))),target_names=CLASSES,zero_division=0))
    test_pred=test_df.copy();test_pred['pred']=pred.tolist();test_pred['confidence']=probs.max(axis=1).tolist()
    test_pred['probabilities']=[p.tolist() for p in probs]
    test_pred.to_csv(OUT/'test_predictions_v3.csv',index=False)
    rows=[]
    for vid,g in test_pred.groupby('video_id'):
        p=np.mean(np.stack(g.probabilities.to_numpy()),axis=0)
        rows.append({'video_id':vid,'truth':g.label.mode().iloc[0],'pred':CLASSES[int(p.argmax())],'confidence':float(p.max()),'n_crops':len(g)})
    video_results=pd.DataFrame(rows);video_results.to_csv(OUT/'test_video_v3.csv',index=False)
    print('\n=== Par vidéo ===')
    print('Accuracy:',round(accuracy_score(video_results.truth,video_results.pred),3))
    print('Macro-F1:',round(f1_score(video_results.truth,video_results.pred,labels=CLASSES,average='macro',zero_division=0),3))
    print(classification_report(video_results.truth,video_results.pred,labels=CLASSES,zero_division=0))
    cm=confusion_matrix(video_results.truth,video_results.pred,labels=CLASSES)
    fig,ax=plt.subplots(figsize=(10,8));ax.imshow(cm,cmap='Blues');ax.set_xticks(range(11),CLASSES,rotation=60);ax.set_yticks(range(11),CLASSES);ax.set_xlabel('Prédit');ax.set_ylabel('Réel');plt.show()
    print('Vidéos de test par espèce:',video_results.truth.value_counts().to_dict())
else:print('Aucun test indépendant : ajoutez des vidéos.')

## 9 — Vidéo libre : détection, classification, suivi et export MP4
**Détection et classification sont séparées**. Le seuil détecteur est volontairement bas pour favoriser le rappel. Le suivi par IoU fournit des estimations, **pas un dénombrement garanti des individus uniques**. Les animaux non reconnus restent « incertain ».

In [ ]:
# Importer une vidéo locale dans Colab ou préciser directement INPUT_VIDEO.
USE_UPLOAD=True
if USE_UPLOAD:
    uploaded=files.upload()
    assert uploaded,'Aucun fichier sélectionné.'
    INPUT_VIDEO=Path('/content')/next(iter(uploaded))
else:
    INPUT_VIDEO=Path('/content/ma_video.mp4')
assert INPUT_VIDEO.exists(),INPUT_VIDEO
print('Vidéo:',INPUT_VIDEO)

In [ ]:
INFER_EVERY_N_FRAMES=2
INFER_DET_CONF=0.10
CLASS_THRESHOLD=0.48  # seuil de rejet à calibrer sur validation
MIN_TRACK_HITS=2
IOU_THRESHOLD=0.18
MAX_MISSES=5
INFER_DIR=WORK/'infer_frames';INFER_DIR.mkdir(exist_ok=True)
for p in INFER_DIR.glob('*.jpg'):p.unlink()
cap=cv2.VideoCapture(str(INPUT_VIDEO));assert cap.isOpened(),'Vidéo illisible'
fps=cap.get(cv2.CAP_PROP_FPS);fps=fps if fps and np.isfinite(fps) and fps>0 else 25
w=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH));h=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT));assert w>0 and h>0
meta=[];i=0
while True:
    ok,frame=cap.read()
    if not ok:break
    if i%INFER_EVERY_N_FRAMES==0:
        path=INFER_DIR/f'infer_{i:07d}.jpg'
        if cv2.imwrite(str(path),frame):meta.append({'frame_idx':i,'time_sec':i/fps,'path':str(path)})
    i+=1
cap.release();assert meta,'Vidéo vide'
print('Frames analysées:',len(meta),'/',i)
# Recharger le meilleur classificateur si la session a changé
if 'model' not in globals():
    ck=torch.load(CHECKPOINT,map_location=DEVICE,weights_only=False)
    model=timm.create_model(ck['model_name'],pretrained=False,num_classes=len(CLASSES)).to(DEVICE)
    model.load_state_dict(ck['state_dict'])
model.eval()
if 'detector' not in globals():detector=run_detector.load_detector(DETECTOR_NAME)

def iou(a,b):
    x=max(a[0],b[0]);y=max(a[1],b[1]);xx=min(a[2],b[2]);yy=min(a[3],b[3])
    inter=max(0,xx-x)*max(0,yy-y)
    area=lambda z:max(0,z[2]-z[0])*max(0,z[3]-z[1])
    return inter/(area(a)+area(b)-inter+1e-9)

tracks={};next_id=1;frame_results=[]
for m in tqdm(meta,desc='Détection + classification'):
    ds=[d for d in detect_image(m['path']) if float(d['conf'])>=INFER_DET_CONF]
    image=Image.open(m['path']).convert('RGB')
    for t in tracks.values():t['misses']+=1
    detections=[]
    for d in ds:
        x,y,bw,bh=d['bbox'];box=[int(x*w),int(y*h),int((x+bw)*w),int((y+bh)*h)]
        crop_path=WORK/'_infer_crop.jpg'
        if not crop_xywh(m['path'],d['bbox'],crop_path):continue
        with Image.open(crop_path) as crop: tensor=val_tf(crop.convert('RGB')).unsqueeze(0).to(DEVICE)
        with torch.no_grad():p=model(tensor).softmax(1)[0].cpu().numpy()
        label=CLASSES[int(p.argmax())] if float(p.max())>=CLASS_THRESHOLD else 'incertain'
        detections.append({'box':box,'bbox':d['bbox'],'label':label,'score':float(p.max()),'det_conf':float(d['conf']),'probs':p.tolist()})
    used=set()
    for d in sorted(detections,key=lambda x:x['det_conf'],reverse=True):
        candidates=[(iou(d['box'],t['box']),tid) for tid,t in tracks.items() if tid not in used and t['misses']<=MAX_MISSES]
        score,tid=max(candidates,default=(0,None))
        if score<IOU_THRESHOLD:
            tid=next_id;next_id+=1
            tracks[tid]={'box':d['box'],'misses':0,'hits':0,'votes':np.zeros(len(CLASSES)), 'first_sec':m['time_sec'],'last_sec':m['time_sec']}
        t=tracks[tid];t['box']=d['box'];t['misses']=0;t['hits']+=1;t['last_sec']=m['time_sec']
        if d['label']!='incertain':t['votes']+=np.array(d['probs'])
        d['track_id']=tid;used.add(tid)
    frame_results.append({'frame_idx':m['frame_idx'],'time_sec':m['time_sec'],'detections':detections})
print('Détections totales:',sum(len(f['detections']) for f in frame_results),'| Tracks:',len(tracks))

In [ ]:
# Consolider les labels par track et produire le MP4 annoté.
track_labels={}
for tid,t in tracks.items():
    v=t['votes'];track_labels[tid]=CLASSES[int(v.argmax())] if t['hits']>=MIN_TRACK_HITS and float(v.sum())>0 else 'incertain'

annotated=OUT/(INPUT_VIDEO.stem+'_annote_v3.mp4')
writer=cv2.VideoWriter(str(annotated),cv2.VideoWriter_fourcc(*'mp4v'),fps,(w,h))
assert writer.isOpened(),'Impossible de créer le MP4'
by_idx={f['frame_idx']:f for f in frame_results}
cap=cv2.VideoCapture(str(INPUT_VIDEO));frame_i=0;current=[]
while True:
    ok,frame=cap.read()
    if not ok:break
    if frame_i in by_idx:current=by_idx[frame_i]['detections']
    for d in current:
        x1,y1,x2,y2=d['box'];label=track_labels.get(d['track_id'],'incertain')
        color=(0,210,0) if label!='incertain' else (0,180,255)
        cv2.rectangle(frame,(x1,y1),(x2,y2),color,2)
        text=f"#{d['track_id']} {label} {d['score']:.2f}"
        cv2.putText(frame,text,(x1,max(18,y1-7)),cv2.FONT_HERSHEY_SIMPLEX,.55,color,2)
    writer.write(frame);frame_i+=1
cap.release();writer.release()

counts=Counter(track_labels[tid] for tid,t in tracks.items() if t['hits']>=MIN_TRACK_HITS and track_labels[tid]!='incertain')
max_sim=Counter()
for f in frame_results:
    visible=Counter(track_labels.get(d['track_id'],'incertain') for d in f['detections'] if tracks[d['track_id']]['hits']>=MIN_TRACK_HITS)
    for s,v in visible.items():
        if s!='incertain':max_sim[s]=max(max_sim[s],v)
report={'video':str(INPUT_VIDEO),'fps':fps,'sample_every_n_frames':INFER_EVERY_N_FRAMES,'detections':sum(len(f['detections']) for f in frame_results),'species':[{'species':s,'latin':LATIN[s],'tracks_estimes':int(counts[s]),'maximum_simultane_estime':int(max_sim[s])} for s in CLASSES if counts[s]],'tracks':[{'id':tid,'species':track_labels[tid],'hits':t['hits'],'first_sec':t['first_sec'],'last_sec':t['last_sec']} for tid,t in tracks.items()],'status':'Animaux non détectés' if not any(f['detections'] for f in frame_results) else ('Animaux détectés, espèces incertaines' if not counts else 'Espèces identifiées')}
report_path=OUT/(INPUT_VIDEO.stem+'_rapport_v3.json');report_path.write_text(json.dumps(report,ensure_ascii=False,indent=2))
print(json.dumps({k:v for k,v in report.items() if k!='tracks'},ensure_ascii=False,indent=2))
print('MP4:',annotated,'\nJSON:',report_path)
display(Video(str(annotated),embed=True,width=720))

## 10 — Diagnostic si la vidéo reste vide
Si `Détections totales = 0`, le problème vient **du détecteur**, pas de la classification. Inspectez des images de la vidéo avec un seuil à 0,05, testez `MDV5B`, vérifiez les images nocturnes et la taille des animaux. Si des boîtes existent mais que les espèces sont « incertaines », inspectez les crops, les probabilités et le seuil de classification. Ne baissez pas les seuils uniquement pour faire apparaître une espèce.

**Limites :** les vidéos externes peuvent avoir des labels incomplets ; les animaux multiples d’espèces différentes dans une même vidéo nécessitent une revue humaine ; l’IoU peut fragmenter un individu en plusieurs tracks. Pour la chouette et le merle, quelques vidéos supplémentaires ne suffisent pas nécessairement à obtenir une estimation fiable. Les fichiers sur `/content` sont temporaires ; les résultats et corrections sont sauvegardés sur Drive.

**Sources API MegaDetector :** [Documentation Python](https://megadetector.readthedocs.io/en/latest/) · [Référence `run_detector`](https://megadetector.readthedocs.io/en/latest/detection.html)